In [1]:
# ==========================================================
# Model Comparison Notebook
# ==========================================================

import warnings
warnings.filterwarnings("ignore")

import os
import json
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

plt.style.use("ggplot")
pd.set_option("display.max_columns", None)

In [2]:
# ==========================================================
# Load Dataset
# ==========================================================

DATA_PATH = "../data/raw/weather.csv"

if not os.path.exists(DATA_PATH):
    DATA_PATH = "../data/raw/Global Weather Repository.csv"

df = pd.read_csv(DATA_PATH)

df["last_updated"] = pd.to_datetime(df["last_updated"])

print(df.shape)

df.head()

(151047, 41)


,country,location_name,latitude,longitude,timezone,last_updated_epoch,last_updated,temperature_celsius,temperature_fahrenheit,condition_text,wind_mph,wind_kph,wind_degree,wind_direction,pressure_mb,pressure_in,precip_mm,precip_in,humidity,cloud,feels_like_celsius,feels_like_fahrenheit,visibility_km,visibility_miles,uv_index,gust_mph,gust_kph,air_quality_Carbon_Monoxide,air_quality_Ozone,air_quality_Nitrogen_dioxide,air_quality_Sulphur_dioxide,air_quality_PM2.5,air_quality_PM10,air_quality_us-epa-index,air_quality_gb-defra-index,sunrise,sunset,moonrise,moonset,moon_phase,moon_illumination
0,Afghanistan,Kabul,34.52,69.18,Asia/Kabul,1715849100,2024-05-16 13:15:00,26.6,79.8,Partly Cloudy,8.3,13.3,338,NNW,1012.0,29.89,0.0,0.00,24,30,25.3,77.5,10.0,6.0,7.0,9.5,15.3,277.0,103.0,1.1,0.2,8.4,26.6,1,1,04:50 AM,06:50 PM,12:12 PM,01:11 AM,Waxing Gibbous,55
1,Albania,Tirana,41.33,19.82,Europe/Tirane,1715849100,2024-05-16 10:45:00,19.0,66.2,Partly cloudy,6.9,11.2,320,NW,1012.0,29.88,0.1,0.00,94,75,19.0,66.2,10.0,6.0,5.0,11.4,18.4,193.6,97.3,0.9,0.1,1.1,2.0,1,1,05:21 AM,07:54 PM,12:58 PM,02:14 AM,Waxing Gibbous,55
2,Algeria,Algiers,36.76,3.05,Africa/Algiers,1715849100,2024-05-16 09:45:00,23.0,73.4,Sunny,9.4,15.1,280,W,1011.0,29.85,0.0,0.00,29,0,24.6,76.4,10.0,6.0,5.0,13.9,22.3,540.7,12.2,65.1,13.4,10.4,18.4,1,1,05:40 AM,07:50 PM,01:15 PM,02:14 AM,Waxing Gibbous,55
3,Andorra,Andorra La Vella,42.50,1.52,Europe/Andorra,1715849100,2024-05-16 10:45:00,6.3,43.3,Light drizzle,7.4,11.9,215,SW,1007.0,29.75,0.3,0.01,61,100,3.8,38.9,2.0,1.0,2.0,8.5,13.7,170.2,64.4,1.6,0.2,0.7,0.9,1,1,06:31 AM,09:11 PM,02:12 PM,03:31 AM,Waxing Gibbous,55
4,Angola,Luanda,-8.84,13.23,Africa/Luanda,1715849100,2024-05-16 09:45:00,26.0,78.8,Partly cloudy,8.1,13.0,150,SSE,1011.0,29.85,0.0,0.00,89,50,28.7,83.6,10.0,6.0,8.0,12.5,20.2,2964.0,19.0,72.7,31.5,183.4,262.3,5,10,06:12 AM,05:55 PM,01:17 PM,12:38 AM,Waxing Gibbous,55


In [3]:
# ==========================================================
# Feature Engineering
# ==========================================================

features = [
    "humidity",
    "pressure_mb",
    "wind_kph",
    "precip_mm",
    "cloud",
    "visibility_km",
    "uv_index",
    "air_quality_PM2.5",
    "air_quality_PM10"
]

df = df.sample(
    n=30000,
    random_state=42
).reset_index(drop=True)

X = df[features]

y = df["temperature_celsius"]

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(24000, 9)
(6000, 9)


In [7]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

In [8]:
def evaluate_model(model_name, y_true, prediction):

    mae = mean_absolute_error(y_true, prediction)
    rmse = np.sqrt(mean_squared_error(y_true, prediction))
    r2 = r2_score(y_true, prediction)

    return {
        "Model": model_name,
        "MAE": round(mae,3),
        "RMSE": round(rmse,3),
        "R2": round(r2,4)
    }

In [9]:
# ==========================================================
# Linear Regression
# ==========================================================

lr = LinearRegression()

lr.fit(X_train, y_train)

lr_pred = lr.predict(X_test)

results = []

results.append(
    evaluate_model(
        "Linear Regression",
        y_test,
        lr_pred
    )
)

In [10]:
# ==========================================================
# Random Forest
# ==========================================================

rf = RandomForestRegressor(
    n_estimators=50,
    max_depth=10,
    random_state=42,
    n_jobs=1
)

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

results.append(
    evaluate_model(
        "Random Forest",
        y_test,
        rf_pred
    )
)

In [11]:
# ==========================================================
# XGBoost
# ==========================================================

xgb = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=1
)

xgb.fit(X_train, y_train)

xgb_pred = xgb.predict(X_test)

results.append(
    evaluate_model(
        "XGBoost",
        y_test,
        xgb_pred
    )
)

In [12]:
# ==========================================================
# LightGBM
# ==========================================================

lgb = LGBMRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    random_state=42
)

lgb.fit(X_train, y_train)

lgb_pred = lgb.predict(X_test)

results.append(
    evaluate_model(
        "LightGBM",
        y_test,
        lgb_pred
    )
)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001606 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1381
[LightGBM] [Info] Number of data points in the train set: 24000, number of used features: 9
[LightGBM] [Info] Start training from score 21.242100
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain:

In [13]:
# ==========================================================
# Comparison Table
# ==========================================================

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="R2",
    ascending=False
).reset_index(drop=True)

results_df

,Model,MAE,RMSE,R2
0,XGBoost,3.848,5.345,0.6898
1,LightGBM,3.874,5.383,0.6854
2,Random Forest,4.011,5.592,0.6604
3,Linear Regression,6.364,7.904,0.3217


In [14]:
# ==========================================================
# Best Model
# ==========================================================

best_model = results_df.iloc[0]

print("=" * 50)
print("🏆 BEST MODEL")
print("=" * 50)

print(f"Model : {best_model['Model']}")
print(f"MAE   : {best_model['MAE']}")
print(f"RMSE  : {best_model['RMSE']}")
print(f"R²    : {best_model['R2']}")

🏆 BEST MODEL
Model : XGBoost
MAE   : 3.848
RMSE  : 5.345
R²    : 0.6898


In [15]:
# ==========================================================
# Save Results
# ==========================================================

os.makedirs("../outputs/results", exist_ok=True)

results_df.to_csv(
    "../outputs/results/model_comparison.csv",
    index=False
)

results_df.to_json(
    "../outputs/results/model_comparison.json",
    orient="records",
    indent=4
)

print("✅ Results exported successfully.")

✅ Results exported successfully.
